# Fábrica de videos · render completo con GPU gratis (Google Colab)

Corre toda la fábrica en Colab con GPU gratis: voz clonada con **Chatterbox** (MIT) y render 3D mucho más rápido.

1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.
2. Ejecuta las celdas en orden.

> Este cuaderno no se ha podido probar desde el entorno de desarrollo (sin acceso a Colab). Si una celda falla, copia el error y lo corregimos.

## 1. Bajar el proyecto
Si el repositorio es **privado**, crea un token en GitHub (Settings → Developer settings → Personal access tokens → Fine-grained, solo lectura de este repo) y pégalo cuando lo pida. Si es público, deja el campo vacío.

In [ ]:
import getpass, os
token = getpass.getpass('Token de GitHub (vacío si el repo es público): ')
rama = 'claude/vigilant-darwin-foo45v'  # cámbiala a 'main' cuando lo unas
url = f'https://{token}@github.com/wilomtz11-ctrl/videosIA.git' if token else 'https://github.com/wilomtz11-ctrl/videosIA.git'
!rm -rf videosIA && git clone -q -b {rama} {url} videosIA
%cd videosIA

## 2. Instalar
Python, Node 22 (para HyperFrames) y dependencias. Tarda unos minutos.

In [ ]:
!pip install -q -r requirements.txt chatterbox-tts
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null
!cd motor3d && npm install --silent && npx playwright install --with-deps chromium > /dev/null

## 3. (Opcional) Tu voz
Sube un **WAV de 10 a 30 segundos** de tu voz (claro, sin música ni ruido). Si no subes nada, se usa la voz de Kokoro del episodio.

In [ ]:
from google.colab import files
subidos = files.upload()  # cancela si no quieres clonar tu voz
referencia = next(iter(subidos), None)
print('Referencia:', referencia)

## 4. Construir el video
Cambia el episodio si quieres otro.

In [ ]:
import yaml, pathlib
episodio = 'episodios/bolivia_mar.yaml'
if referencia:
    ep = yaml.safe_load(open(episodio, encoding='utf-8'))
    ep['voz'] = {'motor': 'chatterbox', 'referencia': referencia}
    episodio = 'episodios/_colab.yaml'
    yaml.safe_dump(ep, open(episodio, 'w', encoding='utf-8'), allow_unicode=True)
!python -m fabrica video {episodio}

## 5. Descargar
Se bajan el video y la descripción con fuentes.

In [ ]:
import glob, os
from google.colab import files
nombre = os.path.splitext(os.path.basename(episodio))[0]
for f in glob.glob(f'salida/{nombre}.mp4') + glob.glob(f'salida/{nombre}_descripcion.txt'):
    files.download(f)